#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birthday_date",
    "GEN": "gender"
}

# Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.erp_cust")

# Data Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            F.regexp_replace(F.trim(F.col(field.name)), r'^NAS', '')
        )

## Renaming the Colums

In [0]:
for old_name, new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Write Into Silver Table

In [0]:
(
    df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable("workspace.silver.erp_cust")
)

In [0]:
%sql
select * from workspace.silver.erp_cust